# CORP-ENV TRL + Unsloth Training Notebook

This notebook is the judge-rerunnable training path for CORP-ENV. It verifies examples, prepares SFT data, runs Unsloth + TRL SFT, optionally runs Unsloth + TRL GRPO, evaluates model stages through the OpenEnv environment, and generates result plots.

Recommended runtime: Colab GPU, Lightning AI H100, or another CUDA GPU machine. For a quick judge smoke run, keep `MAX_STEPS` small.

## 1. Setup

In Colab, set `REPO_URL` to your GitHub or Hugging Face repo URL. If the repo is already mounted/cloned, skip the clone cell and `cd` into the repo.

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # TODO: replace
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
HF_ORG_OR_USER = "<your-user-or-org>"  # TODO: replace
MAX_STEPS = 30  # SFT quick judge smoke; remove or lower for a full-epoch SFT run
GRPO_MAX_STEPS = 150  # used only when RUN_GRPO is True
RUN_GRPO = False  # set True after SFT smoke passes

In [ ]:
!git clone {REPO_URL} corp_gym || true
%cd corp_gym
!pip install -U pip
!pip install -e ".[training,plots]"

## 2. Optional Hugging Face Login

Run this if you want to download private models/datasets or push adapters.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Local OpenEnv Checks

These are lightweight and should pass before training.

In [ ]:
!python -m unittest discover -s tests
!openenv validate

## 4. Verify Examples And Build SFT Data

If `data/raw/e1_m1_examples.jsonl` already exists, the import command is harmless to skip. The verification step replays examples through `CorpEnvironment` and filters bad trajectories.

In [ ]:
!python scripts/import_generated_examples.py \
  --inputs data/raw/e1_to_e100_tasks.py data/raw/m1_to_m100_tasks.py \
  --output data/raw/e1_m1_examples.jsonl

!python scripts/verify_examples.py \
  --input data/raw/e1_m1_examples.jsonl \
  --clean data/processed/e1_m1_clean.jsonl \
  --rejected data/processed/e1_m1_rejected.jsonl \
  --summary results/e1_m1_verification_summary.json \
  --all-records results/e1_m1_verification_all.jsonl

!python scripts/generate_sft_data.py --tasks h1_acquisition_defence --per-task 24 --output data/raw/h1_seed.jsonl
!python scripts/verify_examples.py --input data/raw/h1_seed.jsonl --clean data/processed/h1_seed_clean.jsonl --rejected data/processed/h1_seed_rejected.jsonl
!python scripts/prepare_sft_data.py

## 5. Baseline Eval

This gives a pre-training comparison point.

In [ ]:
!python eval.py --policy scripted_weak --label baseline --output results/baseline_eval.jsonl
!python eval.py --policy oracle --label oracle --output results/oracle_eval.jsonl

## 6. SFT With Unsloth + TRL

This runs `training/train_sft.py`, which uses `unsloth.FastLanguageModel` and `trl.SFTTrainer`. For a real run, increase `MAX_STEPS` or remove the flag and use epochs.

In [ ]:
!python training/train_sft.py \
  --model {BASE_MODEL} \
  --data data/sft/e1_m1_h1_examples.jsonl \
  --output outputs/sft_adapter \
  --max-steps {MAX_STEPS} \
  --push-to-hub {HF_ORG_OR_USER}/corp-env-sft-adapter

## 7. Evaluate SFT Adapter

In [ ]:
!python eval.py \
  --policy hf \
  --label sft \
  --model {BASE_MODEL} \
  --adapter outputs/sft_adapter \
  --output results/sft_eval.jsonl

## 8. Optional GRPO With Unsloth + TRL

This runs `training/train_grpo.py`, which uses `trl.GRPOTrainer` and calls the real CORP-ENV environment reward. Keep this off for the first smoke run.

In [ ]:
if RUN_GRPO:
    !python training/train_grpo.py \
      --model {BASE_MODEL} \
      --adapter outputs/sft_adapter \
      --examples data/processed/e1_m1_clean.jsonl,data/processed/h1_seed_clean.jsonl \
      --output outputs/grpo_adapter \
      --max-steps {GRPO_MAX_STEPS} \
      --dataloader-num-workers 2 \
      --push-to-hub {HF_ORG_OR_USER}/corp-env-grpo-adapter

    !python eval.py \
      --policy hf \
      --label grpo \
      --model {BASE_MODEL} \
      --adapter outputs/grpo_adapter \
      --output results/grpo_eval.jsonl

## 9. Generate Final Plots

In [ ]:
inputs = "results/baseline_eval.jsonl results/sft_eval.jsonl"
if RUN_GRPO:
    inputs += " results/grpo_eval.jsonl"
!python plot_results.py --inputs {inputs} --output-dir results
!ls -lh results